# Quickstart of *TanSacNet Package* (Python)

A brief introduction to the locally structured unitary network (LSUN) with
*TanSacNet Package*.

This notebook is the Python (PyTorch) counterpart of the live script
[`quickstart.m`](quickstart.m). The sections correspond one to one.

## Summary

*TanSacNet* is an abbreviation of *Tangent Space Adaptive Control
Networks*. This package is developed for

* Experiments,
* Development and
* Implementation

of sampling embedded manifold of image and volumetric data as a set of
tangent space bases.

A locally structured unitary network (LSUN) is a lattice-structured
unitary (Parseval tight) transform whose parameters can be changed block by
block. The Python package `torch_tansacnet` provides the LSUN layers and
networks for PyTorch. See [`python/README.md`](python/README.md) for the
details.

## Install the package

Run the following once in this directory (`code`) of the repository:

```sh
pip install "./python[examples]"
```

In [ ]:
import math

import matplotlib.pyplot as plt
import torch

from torch_tansacnet import LsunAnalysis2dNetwork, OrthonormalTransform

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Set the parameters

An LSUN of the following configuration is created:

* Input size: $32\times 32$
* Stride (decimation factor): $4\times 4$, which gives $16$ channels
* Overlapping factor: $3\times 3$
* Number of tree levels: $1$
* No DC leakage, i.e. one vanishing moment

In [ ]:
szx = 32  # Width
szy = 32  # Height
stride = [4, 4]  # Stride [vertical, horizontal]
ovlpFactor = [3, 3]  # Overlapping factor [vertical, horizontal]
nLevels = 1  # Number of tree levels
noDcLeakage = True  # No DC leakage
stdInitAng = math.pi / 6  # Standard deviation of the initial angles
nChs = stride[0] * stride[1]  # Number of channels

## Create the analysis and synthesis networks

`LsunAnalysis2dNetwork` creates the analysis LSUN, which corresponds to
`fcn_createlsunlgraph2d` with `'Mode','Analyzer'` in MATLAB.

The angles of the rotation layers are the learnable parameters. They are
set to random values here to obtain a non-trivial transform.

The synthesis LSUN is the adjoint of the analysis LSUN and obtained by the
property `T`. The parameters are copied from the analysis LSUN, as
`fcn_cpparamsana2syn` in MATLAB.

In [ ]:
torch.manual_seed(0)
analysisnet = LsunAnalysis2dNetwork(
    input_size=[szy, szx],
    stride=stride,
    overlapping_factor=ovlpFactor,
    number_of_levels=nLevels,
    no_dc_leakage=noDcLeakage,
    device=device,
)


def init_angles(m):
    if isinstance(m, OrthonormalTransform):
        torch.nn.init.normal_(m.angles, mean=0.0, std=stdInitAng)


analysisnet.apply(init_angles)
synthesisnet = analysisnet.T

## Confirm the perfect reconstruction

Since LSUN is unitary, the synthesis LSUN reconstructs the input of the
analysis LSUN from its output, i.e.

$$\mathbf{x} = \mathbf{D}\mathbf{D}^T\mathbf{x},$$

where $\mathbf{D}^T$ and $\mathbf{D}$ denote the analysis and synthesis
LSUNs, respectively.

The analysis LSUN of one level outputs the DC and AC coefficients
separately. The images are given as tensors of
$N\times C\times H\times W$ (batch, channel, height, width).

In [ ]:
x = torch.rand(1, 1, szy, szx, device=device)
with torch.no_grad():
    s = analysisnet(x)  # [DC, AC] coefficients
    y = synthesisnet(s)
print("Shapes of the coefficients:", [tuple(c.shape) for c in s])
print(f"MSE: {torch.mean((x - y) ** 2).item():.3e}")

## Show the atomic images

The atomic images (basis images) of LSUN are the columns of
$\mathbf{D}$. They are obtained by synthesizing a unit impulse placed at
the center of each channel of the coefficients.

In [ ]:
cy, cx = szy // stride[0] // 2, szx // stride[1] // 2  # Center of the coefficients
atoms = []
with torch.no_grad():
    for iCh in range(nChs):
        coefs = [torch.zeros_like(c) for c in s]
        if iCh == 0:  # DC channel
            coefs[0][0, cy, cx, 0] = 1
        else:  # AC channels
            coefs[1][0, cy, cx, iCh - 1] = 1
        atoms.append(synthesisnet(coefs).squeeze().cpu())
atoms = torch.stack(atoms)

# Crop the support of the atoms
rows = torch.nonzero(atoms.abs().amax(dim=(0, 2)) > 1e-6).squeeze()
cols = torch.nonzero(atoms.abs().amax(dim=(0, 1)) > 1e-6).squeeze()
atoms = atoms[:, rows[0] : rows[-1] + 1, cols[0] : cols[-1] + 1]

vmax = atoms.abs().max().item()  # Zero is shown in gray
fig, axes = plt.subplots(stride[0], stride[1], figsize=(6, 6))
for iCh, ax in enumerate(axes.flat):
    ax.imshow(atoms[iCh], cmap="gray", vmin=-vmax, vmax=vmax, interpolation="nearest")
    ax.axis("off")
fig.suptitle("Atomic images of LSUN");

## Next steps

Several examples are found under the directory `examples`. For the
training of LSUN as an image approximation like PCA, see
[`examples/lsun/main_lsun.ipynb`](examples/lsun/main_lsun.ipynb).